In [50]:
# https://www.kaggle.com/competitions/petfinder-adoption-prediction/data

import pandas as pd

df = pd.read_csv("train_Animal.csv")

print(df.shape)
df.head()


(14993, 24)


,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,...,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PetID,PhotoAmt,AdoptionSpeed
0,2,Nibble,3,299,0,1,1,7,0,1,...,1,1,100,41326,8480853f516546f6cf33aa88cd76c379,0,Nibble is a 3+ month old ball of cuteness. He ...,86e1089a3,1.0,2
1,2,No Name Yet,1,265,0,1,1,2,0,2,...,1,1,0,41401,3082c7125d8fb66f7dd4bff4192c8b14,0,I just found it alone yesterday near my apartm...,6296e909a,2.0,0
2,1,Brisco,1,307,0,1,2,7,0,2,...,1,1,0,41326,fa90fa5b1ee11c86938398b60abc32cb,0,Their pregnant mother was dumped by her irresp...,3422e4906,7.0,3
3,1,Miko,4,307,0,2,1,2,0,2,...,1,1,150,41401,9238e4f44c71a75282e62f7136c6b240,0,"Good guard dog, very alert, active, obedience ...",5842f1ff5,8.0,2
4,1,Hunter,1,307,0,1,1,0,0,2,...,1,1,0,41326,95481e953f8aed9ec3d16fc4509537e8,0,This handsome yet cute boy is up for adoption....,850a43f90,3.0,2


In [51]:
# 성격/상태/외형 으로 비슷함 계산(similarity)
use_cols = [
    "Type",          # 개/고양이
    "Breed1",        # 품종
    "Gender",
    "Age",
    "MaturitySize",
    "FurLength",
    "Vaccinated",
    "Dewormed",
    "Sterilized",
    "Health",
    "Color1"
]

df_rec = df[use_cols].copy()


In [52]:
# 결측치
df_rec = df_rec.fillna(0)


In [53]:
# 문자열 → 숫자 변환
# 컬럼 1개 → 카테고리 개수만큼 feature 증가
df_encoded = pd.get_dummies(df_rec)

In [54]:
#  정규화
# Age (0~100)
# Vaccinated (0/1) 스케일이 다르면 비슷함 이 깨짐
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded)


In [55]:
# 유사도 계산
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(X_scaled)

In [59]:
def recommend_pet(index, top_k=10, explain=False):
    # index : 기준 동물 번호 (예: 0번 동물)
    # top_k : 추천할 동물 개수 (기본 10마리)
    # explain : 추천 이유 출력 여부

    # similarity : 동물 간 유사도 점수 표
    # 선택한 index 동물과 다른 동물들의 유사도 점수 가져오기
    sim_scores = similarity[index]

    # argsort() : 유사도를 작은 순서대로 정렬 (인덱스 반환)
    # [::-1]     : 뒤집어서 큰 순서(유사도 높은 순)로 변경
    # [1:top_k+1]: 자기 자신(0번, 유사도=1.0) 제외 후 Top-K 선택
    similar_idx = sim_scores.argsort()[::-1][1:top_k+1]

    # 추천된 동물의 정보만 가져오기
    # iloc : 행 번호 기준 데이터 선택
    rec = df.iloc[similar_idx][["Type", "Breed1", "Age", "Gender"]]

    # explain=True일 때 추천 과정 설명 출력
    if explain:
        print("기준 동물 (입력)")
        # 비교 기준이 되는 동물
        print(df_rec.iloc[index][["Type", "Breed1", "Age", "Gender"]])

        print("\n추천 결과 Top-K")
        # 추천된 동물 목록
        print(rec)

        print("\n비슷한 이유 (기준 vs Top1 비교)")
        # 가장 유사한 1순위 동물
        top1 = similar_idx[0]

        # 기준 동물과 추천 1위 동물 feature 비교
        compare = pd.DataFrame({
            "base_pet": df_rec.iloc[index],
            "top1_reco": df_rec.iloc[top1]
        })
        print(compare)

    # 추천 결과 반환
    return rec


In [57]:
recommend_pet(0, top_k=10, explain=True)


기준 동물 (입력)
Type        2
Breed1    299
Age         3
Gender      1
Name: 0, dtype: int64

추천 결과 Top-K
      Type  Breed1  Age  Gender
6583     2     299    2       1
9281     2     299    1       1
2156     2     292    3       1
274      2     292    3       1
9313     2     289    2       1
261      2     266    3       1
4530     2     266    3       1
9260     2     266    3       1
1136     2     266    3       1
1658     2     266    3       1

비슷한 이유 (기준 vs Top1 비교)
              base_pet  top1_reco
Type                 2          2
Breed1             299        299
Gender               1          1
Age                  3          2
MaturitySize         1          1
FurLength            1          1
Vaccinated           2          2
Dewormed             2          2
Sterilized           2          2
Health               1          1
Color1               1          1


,Type,Breed1,Age,Gender
6583,2,299,2,1
9281,2,299,1,1
2156,2,292,3,1
274,2,292,3,1
9313,2,289,2,1
261,2,266,3,1
4530,2,266,3,1
9260,2,266,3,1
1136,2,266,3,1
1658,2,266,3,1


## 🔎 추천 결과 해석

추천 결과를 보면 동일한 Type(종)이 많이 나타나는 것을 확인할 수 있다.

이는 유사도 계산 시 종(species) 정보가 중요한 feature로 사용되기 때문으로 보인다.

---

## 📌 주요 컬럼 의미

### ✔ Type (동물 종류)
- 1 : Dog (강아지)
- 2 : Cat (고양이)

같은 종일수록 feature 값이 유사해지므로
추천 결과에서도 동일한 Type이 많이 나타난다.

---

### ✔ Breed1 (품종 번호)
- 동물의 주 품종(primary breed)을 의미하는 ID 값
- 예: `299 → Domestic Short Hair`

현재는 숫자로 표현되어 있어
추천 결과를 직관적으로 이해하기는 어렵다.

---

### ✔ Gender (성별)
- 1 : Male (수컷)
- 2 : Female (암컷)
- 3 : Mixed / Unknown

성별 또한 유사도 계산에 포함되기 때문에
동일한 성별의 동물이 추천 결과에 자주 나타난다.

---

## 💡 추가 개선 아이디어

`BreedLabels.csv`와 같은 label 파일을 활용하면
품종 ID를 실제 품종 이름으로 변환할 수 있다.

이렇게 하면 추천 결과를 사람이 더 쉽게 해석할 수 있을 것 같다.